In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "FIFA WC Dataset"
CLEANED_DATA = PROJECT_ROOT / "cleaned_data"
PROCESSED_DATA = PROJECT_ROOT / "processed_data"

team_matches = pd.read_csv(PROCESSED_DATA / "team_matches.csv")
saved_stats = pd.read_csv(PROCESSED_DATA / "team_stats.csv")
results = pd.read_csv(CLEANED_DATA / "results_2000_fifa211.csv")

In [2]:
results["date"] = pd.to_datetime(results["date"], dayfirst = True)

In [3]:
# result_played will only store the matches which are already played. Future matches are removed (until they happen).
results_played = results[results["home_score"].notna() & results["away_score"].notna()].copy()

In [4]:
results_played = results_played.sort_values("date").reset_index(drop=True)

In [5]:
results_played.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
23755,2026-07-05,Mexico,England,2,3,FIFA World Cup,Mexico City,Mexico,False
23756,2026-07-06,Portugal,Spain,0,1,FIFA World Cup,Arlington,United States,True
23757,2026-07-06,USA,Belgium,1,4,FIFA World Cup,Seattle,United States,False
23758,2026-07-07,Argentina,Egypt,3,2,FIFA World Cup,Atlanta,United States,True
23759,2026-07-07,Switzerland,Colombia,0,0,FIFA World Cup,Vancouver,Canada,True


In [6]:
INITIAL_ELO = 1500

elo_ratings = {}

history_records = []

In [7]:
def get_k_value(tournament):

    if tournament == "Friendly":
        return 10

    elif tournament == "FIFA World Cup":
        return 60

    elif tournament == "FIFA World Cup qualification":
        return 40

    elif tournament in [
        "UEFA Euro",
        "Copa América",
        "AFC Asian Cup",
        "African Cup of Nations",
        "CONCACAF Gold Cup",
        "OFC Nations Cup"
    ]:
        return 35

    elif "qualification" in tournament.lower():
        return 25

    elif "nations league" in tournament.lower():
        return 25

    else:
        return 20
    
elo_ratings = {}

# Created a get_rating function which will get the elo ratings of a team 

def get_rating(team):
    if team not in elo_ratings:
        elo_ratings[team] = INITIAL_ELO
    return elo_ratings[team]

# Created a new get_result function which will store the results Win, Draw, Lose, i.e. W, D, L based on the home-away scores.

def get_result(goals_for, goals_against):
    if goals_for > goals_against:
        return "W"
    elif goals_for < goals_against:
        return "L"
    else:
        return "D"

In [8]:
first_match = results_played.iloc[0]

first_match

date          2000-01-04 00:00:00
home_team                   Egypt
away_team                    Togo
home_score                      2
away_score                      1
tournament               Friendly
city                        Aswan
country                     Egypt
neutral                     False
Name: 0, dtype: object

In [9]:
home_team = first_match["home_team"]
away_team = first_match["away_team"]

home_score = first_match["home_score"]
away_score = first_match["away_score"]

print(home_team, home_score)
print(away_team, away_score)

Egypt 2
Togo 1


In [10]:
home_rating = get_rating(home_team)
away_rating = get_rating(away_team)

print(home_rating)
print(away_rating)

1500
1500


In [11]:
# Calculation if teams draw a match.

expected_home = 1 / (
    1 + 10 ** ((away_rating - home_rating) / 400)
)

expected_away = 1 - expected_home

print(expected_home)
print(expected_away)

0.5
0.5


In [12]:
# Giving points based on match results. if a team Wins it gets 1 point, a Lose gets 0 and 0.5 for a Draw.

if home_score > away_score:
    actual_home = 1
    actual_away = 0

elif home_score < away_score:
    actual_home = 0
    actual_away = 1

else:
    actual_home = 0.5
    actual_away = 0.5

print(actual_home)
print(actual_away)

1
0


In [13]:
def goal_difference_multiplier(goal_difference):
    """
    Returns the Elo multiplier based on goal difference.
    """

    if goal_difference == 1:
        return 1.00
    elif goal_difference == 2:
        return 1.50
    elif goal_difference == 3:
        return 1.75
    elif goal_difference == 4:
        return 2.00
    else:
        return 2.25

In [14]:
# Processing matches in chronological order.

for _, match in results_played.iterrows():

    # Match details
    date = match["date"]
    home_team = match["home_team"]
    away_team = match["away_team"]
    tournament = match["tournament"]
    # Get K value for this tournament
    k = get_k_value(tournament)

    home_score = match["home_score"]
    away_score = match["away_score"]

        # Goal difference multiplier
    goal_difference = abs(home_score - away_score)
    gd_multiplier = goal_difference_multiplier(goal_difference)

    # Current ratings
    home_rating = get_rating(home_team)
    away_rating = get_rating(away_team)

    # Expected scores
    expected_home = 1 / (
        1 + 10 ** ((away_rating - home_rating) / 400)
    )
    expected_away = 1 - expected_home

    # Actual scores
    if home_score > away_score:
        actual_home = 1
        actual_away = 0

    elif home_score < away_score:
        actual_home = 0
        actual_away = 1

    else:
        actual_home = 0.5
        actual_away = 0.5

    # New ratings
    new_home_rating = (
        home_rating +
        k * gd_multiplier * (actual_home - expected_home)
    )

    new_away_rating = (
        away_rating +
        k * gd_multiplier * (actual_away - expected_away)
    )

    # Results
    home_result = get_result(home_score, away_score)
    away_result = get_result(away_score, home_score)

    # Home team history row
    history_records.append({
        "date": date,
        "team": home_team,
        "opponent": away_team,
        "tournament": tournament,
        "goals_for": home_score,
        "goals_against": away_score,
        "result": home_result,
        "elo_before": home_rating,
        "elo_after": new_home_rating
    })

    # Away team history row
    history_records.append({
        "date": date,
        "team": away_team,
        "opponent": home_team,
        "tournament": tournament,
        "goals_for": away_score,
        "goals_against": home_score,
        "result": away_result,
        "elo_before": away_rating,
        "elo_after": new_away_rating
    })

    # Update ratings dictionary
    elo_ratings[home_team] = new_home_rating
    elo_ratings[away_team] = new_away_rating

In [15]:
elo_history_v2_5 = pd.DataFrame(history_records)

In [16]:
list(elo_ratings.items())[:10]

[('Egypt', 1786.1940061348052),
 ('Togo', 1453.205444057514),
 ('Tunisia', 1590.1473958792844),
 ('Trinidad and Tobago', 1527.8819336982708),
 ('Canada', 1766.4920066858415),
 ('Burkina Faso', 1625.7220538219265),
 ('Gabon', 1545.724033822103),
 ('Guatemala', 1617.0244911300558),
 ('Armenia', 1388.802715996043),
 ("Côte d'Ivoire", 1773.083825179261)]

In [17]:
elo_ratings_v2_5 = pd.DataFrame(
    elo_ratings.items(),
    columns=["team", "elo_rating"]
)

elo_ratings_v2_5 = elo_ratings_v2_5.sort_values(
    by="elo_rating",
    ascending=False
)
elo_ratings_v2_5 = elo_ratings_v2_5.sort_values(
    by="elo_rating",
    ascending=False
).reset_index(drop=True)

In [18]:
elo_ratings_v2_5.columns

Index(['team', 'elo_rating'], dtype='str')

In [19]:
# Round history
elo_history_v2_5["elo_before"] = elo_history_v2_5["elo_before"].round(2)
elo_history_v2_5["elo_after"] = elo_history_v2_5["elo_after"].round(2)

# Round final ratings
elo_ratings_v2_5["elo_rating"] = elo_ratings_v2_5["elo_rating"].round(2)

# Add ranking
elo_ratings_v2_5.insert(
    0,
    "rank",
    range(1, len(elo_ratings_v2_5) + 1)
)

In [20]:
print(elo_ratings_v2_5.head())
print(elo_history_v2_5.head())

   rank       team  elo_rating
0     1  Argentina     2070.35
1     2     France     2058.83
2     3      Spain     2034.83
3     4    England     1992.75
4     5    Morocco     1969.34
        date                 team opponent tournament  goals_for  \
0 2000-01-04                Egypt     Togo   Friendly          2   
1 2000-01-04                 Togo    Egypt   Friendly          1   
2 2000-01-07              Tunisia     Togo   Friendly          7   
3 2000-01-07                 Togo  Tunisia   Friendly          0   
4 2000-01-08  Trinidad and Tobago   Canada   Friendly          0   

   goals_against result  elo_before  elo_after  
0              1      W      1500.0    1505.00  
1              2      L      1500.0    1495.00  
2              0      W      1500.0    1511.09  
3              7      L      1495.0    1483.91  
4              0      D      1500.0    1500.00  


In [21]:
elo_ratings_v2_5.to_csv(PROCESSED_DATA / "elo_ratings_v2_5.csv", index = False)
elo_history_v2_5.to_csv(PROCESSED_DATA / "elo_history_v2_5.csv", index = False)

In [22]:
elo_ratings_v2_5.head(10)

,rank,team,elo_rating
0,1,Argentina,2070.35
1,2,France,2058.83
2,3,Spain,2034.83
3,4,England,1992.75
4,5,Morocco,1969.34
5,6,Colombia,1938.29
6,7,Brazil,1935.09
7,8,Netherlands,1933.03
8,9,Norway,1927.77
9,10,Mexico,1922.56


In [23]:
elo_ratings_v2_5.tail(10)

,rank,team,elo_rating
200,201,Turks and Caicos Islands,1065.12
201,202,Bahamas,1063.13
202,203,Brunei Darussalam,1059.26
203,204,US Virgin Islands,1054.42
204,205,Anguilla,1038.30
205,206,Liechtenstein,1020.46
206,207,Timor-Leste,1017.30
207,208,Bhutan,1015.83
208,209,Macau,1008.15
209,210,San Marino,944.08


## V2.5 Summary

Changes introduced:
- Tournament weighting retained from V2.
- Goal Difference (GD) multiplier added to Elo updates.

Goal Difference Multiplier:

| Goal Difference | Multiplier |
|-----------------|-----------:|
| 1 | 1.00 |
| 2 | 1.50 |
| 3 | 1.75 |
| 4 | 2.00 |
| 5+ | 2.25 |

Outcome:
- Elo ratings remain zero-sum (mean ≈ 1500).
- Rating spread increased, rewarding dominant victories while penalizing heavy defeats more than in V2.